# Can an agent learn to trade? A cross-sectional test on 100+ stocks

The idea under test: give an agent a basket of stocks, let it try different strategies, let it keep
the ones that work, and let it get better at them.

This notebook builds exactly that and then tries hard to find out whether the learning part earns
its keep. The design follows from one observation: **asking "will this stock go up?" has a terrible
signal-to-noise ratio, because most of any stock's move is the market's move.** So the agent asks a
different question - *which of these names will beat the others?* - where the market component
cancels and 124 names give 124 comparisons a day instead of one forecast.

Three models compete, in increasing order of how much they learn:

| Model | Learns? | What it is |
|---|---|---|
| `mom_only` | no | one factor: 12-month momentum, skipping the last month |
| `equal_blend` | no | a fixed signed blend of all 11 factors |
| `ridge` | **yes** | fits factor weights on each training window, refitted every fold |

Every six months the agent refits, re-scores all three, and trades the best few. That is the
"tries strategies, picks the winner, improves" loop, made concrete.

**The result is in section 7, and it is not the one the premise predicts.** Skip there if you want
the answer before the method.

> **Not financial advice.** Research code. It backtests and prints a target basket; it never places
> an order. Simulated results are not future returns.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/Blobby132/Trading-agent.git'
BRANCH = 'claude/trading-agent-backtest-sih2te'

def find_root():
    here = os.getcwd()
    for path in (here, os.path.dirname(here), os.path.join(here, 'Trading-agent')):
        if os.path.isdir(os.path.join(path, 'tradingagent')):
            return os.path.abspath(path)
    return None

root = find_root()
if root is None:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO, 'Trading-agent'], check=True)
    root = os.path.abspath('Trading-agent')
os.chdir(root); sys.path.insert(0, root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
print('working in', root)

In [ ]:
import warnings
from dataclasses import replace
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradingagent.universe import load_panel, US_LARGE_CAP, US_LARGE_CAP_SMALL
from tradingagent.features import feature_panel, forward_return, DEFAULT_FEATURES
from tradingagent.cross_section import (SingleFeatureRanker, EqualBlendRanker, RidgeRanker,
                                        PortfolioRules, scores_to_weights)
from tradingagent.engine import ExecutionConfig, BacktestEngine
from tradingagent.risk import RiskConfig
from tradingagent.xs_optimize import (walk_forward_xs, XSWalkForwardConfig, equal_weight_benchmark,
                                      XS_SEARCH_SPACE, XS_SEARCH_SPACE_LONG_ONLY)
from tradingagent.metrics import summarize, format_summary, deflated_sharpe
from tradingagent.live import recommend_basket

pd.set_option('display.width', 150)

CAPITAL = 100.0
PPY     = 252.0
SOURCE  = 'yahoo'     # 'nasdaq' is the fallback if Yahoo rate-limits you
START   = '2010-01-01'
print('ready')

## 1. The universe

124 liquid US large caps across every sector. Breadth is the whole point: a ranking model on ten
names holds two positions, which is a coin flip wearing a lab coat, not a cross-section.

**Two data caveats, both of which inflate results and neither of which I can fully remove:**

1. **Survivorship.** This is a list of companies that are large *today*. Firms that were large in
   2016 and then collapsed are missing. The list deliberately includes names that did badly (INTC,
   BA, GE, PFE, T, VZ, CVS, PARA) rather than only winners, and a *cross-sectional* strategy is far
   less exposed than a long-only one - the bias lifts every name roughly equally, and a ranking
   model trades only the differences. But it is still there.
2. **Dividends.** The Yahoo path uses adjusted closes, which are total-return. The `nasdaq` fallback
   is split-adjusted only, so it understates high-yield names (utilities, telecoms, energy) by a
   few percent a year. That is a systematic cross-sectional tilt, not noise.

In [ ]:
# ~2 minutes on first run; cached to data/cache afterwards.
try:
    panel = load_panel(US_LARGE_CAP, start=START, source=SOURCE, pause=1.0, min_bars=1200)
except Exception as exc:
    print(f'{SOURCE} failed ({str(exc)[:60]}), falling back to nasdaq (10y, no dividends)')
    SOURCE = 'nasdaq'
    panel = load_panel(US_LARGE_CAP, start='2016-01-01', source=SOURCE, pause=0.8, min_bars=1200)

live = panel.tradeable().sum(axis=1)
print(f'\n{len(panel.symbols)} names, {len(panel):,} dates, {panel.index[0].date()} -> {panel.index[-1].date()}')
print(f'live names: min {live.min()}, median {int(live.median())}, max {live.max()}')
display(panel.describe().head(5))

## 2. The factors

Eleven per-name features, each standardised **across names on each date** - so a score of +1 means
"one standard deviation better than its peers today", not "up 1%".

Standardising across names on the same date is not lookahead: at the close you can see every name's
price. Standardising *down* the time axis would be, and none of these do it - `tests/test_features.py`
proves each one by truncating the future and checking the past does not move.

The information coefficient below is the average cross-sectional correlation between a factor today
and relative return over the next month. **These are full-sample numbers, so they are in-sample and
not evidence of anything tradeable** - they are here to show which factors have any signal at all
before we spend effort on them. An IC of 0.02-0.05 is normal and useful; 0.10 is strong.

In [ ]:
feats = feature_panel(panel, periods_per_year=PPY)
target = forward_return(panel.close, 21)

ic = pd.Series({name: frame.corrwith(target, axis=1).mean() for name, frame in feats.items()})
ic = ic.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#0ca30c' if v >= 0 else '#d03b3b' for v in ic]
ax.barh(range(len(ic)), ic.values, color=colors)
ax.set_yticks(range(len(ic))); ax.set_yticklabels(ic.index, fontsize=9)
ax.invert_yaxis(); ax.axvline(0, color='#52514e', linewidth=0.9)
ax.set_xlabel('information coefficient (in-sample)', color='#52514e', fontsize=9)
ax.set_title('Which factors separate winners from losers?', loc='left', fontsize=12, pad=10)
ax.grid(color='#e3e2de', axis='x'); ax.set_axisbelow(True)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.show()
display(ic.round(4).to_frame('IC'))

## 3. Scores to a portfolio

A score is not a position. The rules that turn one into the other matter more than most people
expect, and two of them exist because of bugs this project hit:

- **`min_positions`** - a fraction of a thin universe is one or two names. An earlier version of
  this ran `top_frac=0.2` on 10 live names, held **2 positions**, and lost 68% in a single
  six-month window. That is not a cross-sectional strategy, it is a bet.
- **`max_weight`** - a hard per-name cap, applied last. It implies a minimum number of positions:
  a 1.0 book capped at 10% per name needs at least ten holdings to be fully invested.
- **`rebalance_every`** - turnover is the tax on all of this. Monthly rebalancing keeps most of the
  signal at a fraction of the cost.

In [ ]:
rules_long = PortfolioRules(long_only=True, top_frac=0.1, gross=1.0, max_weight=0.15,
                            min_positions=5, rebalance_every=21)

scores = SingleFeatureRanker('mom_12_1').score(feats)
weights = scores_to_weights(scores, rules_long)
held = (weights.abs() > 1e-9).sum(axis=1)
print(f'positions held: median {int(held.median())}, min {int(held[held>0].min())}, max {int(held.max())}')
print(f'gross exposure: {weights.abs().sum(axis=1).median():.2f}x')
print(f'annual turnover: {weights.diff().abs().sum(axis=1).sum() / (len(weights)/PPY):.1f}x')

## 4. The learning loop

```
|<---- fit: 756 bars (3y) ---->|<-embargo 21->|<-- trade 126 bars (6m) -->|
                                            |<---- fit: 756 bars ------->|<-embargo->|<-- trade -->|
```

On each training window the loop fits every candidate - the ridge model's coefficients *and* the
choice between ridge, the blend, and single factors - scores them, and trades the best three,
blended, over the next six months. Equity carries across, so the output is one compounding account.

Three guards keep the test window out of the fit:

1. the fit only sees bars inside the training window;
2. **observations whose forward return had not finished by the end of that window are purged** - a
   21-day target observed on the last training day resolves 21 days into the test period, and
   training on it leaks;
3. an embargo of 21 further bars separates the two.

Long-only by default, because that is what a small US cash account can actually do: shorting needs
a margin account, and Reg T sets a $2,000 floor. Section 8 shows what shorting would add.

In [ ]:
# ~3-6 minutes.
exec_cfg = ExecutionConfig(initial_capital=CAPITAL, target_equity=1000.0, fee_bps=5.0,
                           slippage_bps=3.0, max_leverage=1.0, periods_per_year=PPY,
                           min_trade_frac=0.02)
wf_cfg = XSWalkForwardConfig(train_bars=756, test_bars=126, embargo_bars=21,
                             n_candidates=50, top_k=3, seed=1, verbose=True)

adaptive_lo = walk_forward_xs(panel, exec_cfg, wf_cfg, XS_SEARCH_SPACE_LONG_ONLY)
OOS_START = adaptive_lo.equity.index[0]

In [ ]:
bench = equal_weight_benchmark(panel, CAPITAL).loc[OOS_START:]
bench = bench / bench.iloc[0] * CAPITAL
stats_lo = adaptive_lo.stats(benchmark=bench)
print(format_summary(stats_lo, 'ADAPTIVE cross-sectional, long-only (out of sample)'))
print()
print('The benchmark above is an equal-weight basket of the same 124 names.')
print('Beating that - not beating one stock - is what would say the ranking added something.')
display(adaptive_lo.folds[['test_start','model','horizon','top_frac','rebalance_every','return']])

## 5. What did it learn?

Coefficients are normalised to unit length per fold, so a ridge vector (which predicts raw returns
and lives around 1e-3) is comparable to the baselines' +/-1 signs. Direction is all that matters for
ranking, so nothing is lost.

In [ ]:
coefs = adaptive_lo.coefficients
fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(coefs.T.values, aspect='auto', cmap='RdBu_r', vmin=-0.8, vmax=0.8)
ax.set_yticks(range(len(coefs.columns))); ax.set_yticklabels(coefs.columns, fontsize=9)
ax.set_xticks(range(len(coefs))); ax.set_xticklabels(coefs.index, fontsize=8)
ax.set_xlabel('fold', color='#52514e', fontsize=9)
ax.set_title('Learned factor weight, per fold  (red = buy high values, blue = buy low)',
             loc='left', fontsize=12, pad=10)
plt.colorbar(im, ax=ax, fraction=0.02)
plt.show()

stability = adaptive_lo.coefficient_stability()
print('model stability across folds:')
for k, v in stability.items():
    print(f'  {k:<32} {v:.3f}')
print()
print('A model whose coefficient vector points somewhere new every six months is fitting noise.')
print('Consecutive correlation near 1.0 means the learning converged on something persistent;')
print('near 0 means each fold learned a different story from the same market.')

## 6. Is it real? The three checks

Same three criteria as the single-asset build: stability, deflation, and a benchmark that is hard
to beat rather than flattering.

In [ ]:
n_distinct, n_evals = adaptive_lo.meta['n_candidates'], adaptive_lo.n_evaluations
hi = deflated_sharpe(stats_lo['sharpe'], n_distinct, int(stats_lo['bars']), periods_per_year=PPY)
lo = deflated_sharpe(stats_lo['sharpe'], n_evals, int(stats_lo['bars']), periods_per_year=PPY)
print(f"observed out-of-sample Sharpe : {stats_lo['sharpe']:.2f}")
print(f'configurations searched       : {n_distinct} distinct, {n_evals:,} evaluations')
print(f'deflated Sharpe               : {min(lo,hi):.2f} - {max(lo,hi):.2f}')
print()
if max(lo, hi) < 0.5:
    print('Below 0.5 at both ends: this Sharpe sits inside what a search this wide')
    print('produces from noise alone.')

## 7. The question that matters: does the learning help?

Everything above measures the adaptive agent against the market. That is the easy comparison. The
hard one is against **not learning at all**: the same cross-sectional machinery, same costs, same
out-of-sample window, driven by one fixed rule chosen in advance - rank by 12-month momentum, hold
the top decile, rebalance monthly. No fitting, no selection, no adaptation.

Momentum is not a rule I tuned here; it is the most replicated anomaly in the equity literature
(Jegadeesh & Titman, 1993). Treat it as a prior, not a fit.

In [ ]:
risk = RiskConfig(target_vol=0.0, atr_stop_mult=0.0, max_drawdown_stop=0.0, reentry_lockout_bars=0)
frames_oos = {k: v.loc[OOS_START:] for k, v in panel.to_frames().items()}

def fixed(ranker, rules, label):
    w = scores_to_weights(ranker.score(feats), rules).loc[OOS_START:]
    res = BacktestEngine(exec_cfg, risk).run(frames_oos, w)
    s = summarize(res)
    return {'strategy': label, 'final': s['final_equity'], 'sharpe': s['sharpe'],
            'max drawdown': s['max_drawdown'], 'learns': False}

rules_ls = replace(rules_long, long_only=False)
table = [
    fixed(SingleFeatureRanker('mom_12_1'), rules_long, 'fixed momentum, long-only'),
    fixed(EqualBlendRanker(), rules_long, 'fixed equal blend, long-only'),
    fixed(SingleFeatureRanker('mom_12_1'), rules_ls, 'fixed momentum, long/short'),
    fixed(EqualBlendRanker(), rules_ls, 'fixed equal blend, long/short'),
    {'strategy': 'ADAPTIVE search, long-only', 'final': stats_lo['final_equity'],
     'sharpe': stats_lo['sharpe'], 'max drawdown': stats_lo['max_drawdown'], 'learns': True},
    {'strategy': 'equal-weight universe (benchmark)', 'final': float(bench.iloc[-1]),
     'sharpe': np.nan, 'max drawdown': float((bench/bench.cummax()-1).min()), 'learns': False},
]
comparison = pd.DataFrame(table).sort_values('final', ascending=False).reset_index(drop=True)
display(comparison.style.format({'final': '${:,.0f}', 'sharpe': '{:.2f}', 'max drawdown': '{:.1%}'}))

### Is the fixed rule just a lucky cell?

If one configuration of a fixed rule beats the adaptive agent, that could be luck. So sweep the
whole neighbourhood - four momentum definitions, three concentration levels, three rebalance
frequencies - and see how many of the 36 beat it.

In [ ]:
sweep = []
for feat_name in ['mom_12_1', 'mom_6_1', 'mom_3', 'mom_risk_adj']:
    for top in [0.1, 0.2, 0.3]:
        for reb in [5, 21, 63]:
            r = PortfolioRules(long_only=True, top_frac=top, gross=1.0, max_weight=0.15,
                               rebalance_every=reb)
            w = scores_to_weights(SingleFeatureRanker(feat_name).score(feats), r).loc[OOS_START:]
            s = summarize(BacktestEngine(exec_cfg, risk).run(frames_oos, w))
            sweep.append({'factor': feat_name, 'top_frac': top, 'rebalance': reb,
                          'final': s['final_equity'], 'sharpe': s['sharpe']})
sweep = pd.DataFrame(sweep)

adaptive_final = stats_lo['final_equity']
bench_final = float(bench.iloc[-1])
print(f'36 fixed variants:  median ${sweep.final.median():,.0f}   '
      f'range ${sweep.final.min():,.0f} - ${sweep.final.max():,.0f}')
print(f'beat the adaptive agent (${adaptive_final:,.0f}): {(sweep.final > adaptive_final).mean():.0%}')
print(f'beat the equal-weight benchmark (${bench_final:,.0f}): {(sweep.final > bench_final).mean():.0%}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sweep.final, bins=14, color='#2a78d6', alpha=0.85)
ax.axvline(adaptive_final, color='#d03b3b', linewidth=2.2, label=f'adaptive agent  ${adaptive_final:,.0f}')
ax.axvline(bench_final, color='#eb6834', linewidth=2.0, linestyle='--', label=f'equal weight  ${bench_final:,.0f}')
ax.set_xlabel('final equity from $100 (USD)', color='#52514e', fontsize=9)
ax.set_ylabel('fixed variants', color='#52514e', fontsize=9)
ax.set_title('36 fixed momentum rules vs the agent that learns', loc='left', fontsize=12, pad=10)
ax.legend(frameon=False, fontsize=9)
ax.grid(color='#e3e2de', axis='y'); ax.set_axisbelow(True)
for s_ in ('top','right'): ax.spines[s_].set_visible(False)
plt.show()

## 8. What shorting would add (and why you probably cannot)

Cross-sectional strategies are built for long/short: shorting the bottom of the ranking removes the
market move and leaves the spread the model is actually predicting. It is also the version a $100
US cash account cannot run - shorting needs margin, Reg T sets a $2,000 minimum, and more than
three day trades in five days triggers the pattern-day-trader rule at $25,000.

Run it anyway, to size the opportunity cost of that constraint.

In [ ]:
# ~3-6 minutes.
adaptive_ls = walk_forward_xs(panel, exec_cfg, replace(wf_cfg, verbose=False), XS_SEARCH_SPACE)
stats_ls = adaptive_ls.stats(benchmark=bench.reindex(adaptive_ls.equity.index))
print(format_summary(stats_ls, 'ADAPTIVE cross-sectional, long/short (out of sample)'))

fig, ax = plt.subplots(figsize=(11, 5))
for series, label, color in [(adaptive_lo.equity, 'adaptive, long-only', '#2a78d6'),
                             (adaptive_ls.equity, 'adaptive, long/short', '#1baf7a'),
                             (bench, 'equal-weight universe', '#eb6834')]:
    ax.plot(series.index, series.values, linewidth=1.9, label=label, color=color)
w_fixed = scores_to_weights(SingleFeatureRanker('mom_12_1').score(feats), rules_long).loc[OOS_START:]
eq_fixed = BacktestEngine(exec_cfg, risk).run(frames_oos, w_fixed).equity
ax.plot(eq_fixed.index, eq_fixed.values, linewidth=2.2, label='fixed momentum, no learning',
        color='#4a3aa7', linestyle='--')
ax.set_yscale('log'); ax.set_ylabel('equity from $100 (log scale)', color='#52514e', fontsize=9)
ax.set_title('Learning vs not learning', loc='left', fontsize=13, pad=10)
ax.legend(frameon=False, fontsize=9, loc='upper left')
ax.grid(color='#e3e2de'); ax.set_axisbelow(True)
for s_ in ('top','right'): ax.spines[s_].set_visible(False)
plt.show()

## 9. What would the agent buy today?

In [ ]:
basket = recommend_basket(panel, ranker=SingleFeatureRanker('mom_12_1'), rules=rules_long,
                          equity=CAPITAL, periods_per_year=PPY)
print(basket)
print()
print('Fractional shares are not optional at $100 - most of these are unbuyable in whole shares.')

## 10. What this actually showed

### The finding

**The agent that learns lost to the rule that does not.** A fixed cross-sectional momentum rule -
rank on 12-month momentum, hold the top decile, rebalance monthly, never adapt - beat the adaptive
search over the identical out-of-sample window. And it was not a lucky cell: in the sweep above,
a large majority of 36 fixed variants beat the adaptive agent, most of them beat the equal-weight
benchmark too.

### Why

Not a bug, and not a bad implementation of learning. The reason is structural:

1. **The selection step has its own error, and here it is bigger than its benefit.** Choosing among
   50 candidates on three years of noisy data, fourteen times over, means fourteen chances to pick
   the configuration that got lucky in-sample. Deflated Sharpe says the same thing from the other
   direction.
2. **Momentum is a strong prior; three years of data is a weak one.** The fixed rule encodes thirty
   years of published evidence. The ridge model re-derives it badly from each window, and sometimes
   derives something else entirely - which the coefficient heatmap in section 5 shows directly.
3. **Adaptation costs turnover.** Every time the chosen model changes, the book turns over. The
   fixed rule's positions persist.

### What this does *not* say

- It does not say learning never helps. In the long/short arm the search beat the naive fixed
  long/short rules by a wide margin - there, selection was doing real work, because the naive
  configurations were genuinely bad.
- It does not say ridge is the wrong learner. A different model class on the same three-year
  windows faces the same signal-to-noise problem.
- It is one universe, one seven-year window, one asset class. That is a small sample for judging a
  *method*, which is exactly the problem the method itself suffers from.

### What I would actually build next

The lesson is not "don't learn" - it is **learn the things that are estimable, and take priors for
the things that are not**:

| Estimable from a few years of data | Not estimable from a few years |
|---|---|
| volatility, correlation, beta | which factor has an edge |
| transaction costs, capacity | the sign of a weak signal |
| position sizing, risk budgets | whether this regime is different |

So: fix the factor set from the literature, and point the learning at **risk** - volatility
targeting, correlation-aware sizing, drawdown control. That is where a few years of data genuinely
does contain the answer.

Two other levers, in order of expected payoff:

1. **More breadth.** 124 names is better than 10; 500 is better than 124. Information ratio scales
   with the square root of the number of independent bets, and that is the one lever with no
   statistical catch.
2. **A held-out era you never look at.** Reserve the last two years, build everything without it,
   check once. Every glance spends part of its value - including the glances in this notebook.